# LLM 모델 비교 실험

이 노트북은 같은 RAG 인덱스와 같은 질문 세트로 LLM만 바꿔 비교한다.

- 1차 후보: `hf.co/unsloth/gemma-4-E4B-it-qat-GGUF:UD-Q4_K_XL`
- 기준선: `gemma3:4b-it-q4_K_M`
- 임베딩: `bge-m3` 고정
- Vector DB: ChromaDB 고정

각 모델마다 FastAPI 서버를 별도 포트에서 실행하고 `/chat`을 호출한 뒤 결과를 `notebooks/results/`에 저장한다.

In [ ]:
import csv
import json
import os
import subprocess
import time
from pathlib import Path
from pprint import pprint

import requests

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULT_DIR = PROJECT_ROOT / "notebooks" / "results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

PYTHON = os.environ.get("PYTHON", "python")
API_PORT = 8010
BASE_URL = f"http://127.0.0.1:{API_PORT}"

MODELS = [
    "hf.co/unsloth/gemma-4-E4B-it-qat-GGUF:UD-Q4_K_XL",
    "hf.co/mradermacher/supergemma4-e4b-abliterated-i1-GGUF:Q4_K_M",
    "gemma3:4b-it-q4_K_M",
]

COMMON_ENV = {
    "OLLAMA_EMBED_MODEL": "bge-m3",
    "CHROMA_PATH": "./data/vector_store/chroma",
    "CHROMA_COLLECTION": "manual_documents",
    "TOP_K": "5",
    "LLM_TEMPERATURE": "0.2",
    "LLM_NUM_PREDICT": "256",
    "LLM_THINK": "false",
}

## 질문 세트

실제 관광 데이터가 `data/raw/`에 들어간 뒤에는 질문을 20개 이상으로 늘린다. 현재 기본 문서가 FAQ라서, 아래에는 관광형 질문과 근거 없음 질문을 함께 둔다.

In [ ]:
QUESTIONS = [
    "환불은 언제까지 가능한가요?",
    "고객 지원 운영 시간은 언제인가요?",
    "설치 후 앱을 어떻게 해야 하나요?",
    "대전 근처에서 역사 관련 관광지를 추천해줘.",
    "아이와 함께 가기 좋은 실내 관광지를 추천해줘.",
    "문서에 없는 숙소 가격 정보를 알려줘.",
    "추천 이유와 출처를 함께 알려줘.",
    "비 오는 날 갈 만한 관광 코스를 알려줘.",
]

## 인덱스 재생성

`bge-m3`로 임베딩 모델을 바꿨다면 반드시 한 번 실행한다.

In [ ]:
def run_rebuild_index():
    completed = subprocess.run(
        [PYTHON, "scripts/rebuild_index.py"],
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
        check=False,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError("rebuild_index.py failed")

# 필요할 때 주석을 해제한다.
# run_rebuild_index()

## 서버 실행 헬퍼

In [ ]:
def wait_for_health(base_url, timeout_sec=60):
    started = time.perf_counter()
    last_error = None
    while time.perf_counter() - started < timeout_sec:
        try:
            response = requests.get(f"{base_url}/health", timeout=2)
            if response.status_code == 200:
                return response.json()
        except Exception as exc:
            last_error = exc
        time.sleep(1)
    raise RuntimeError(f"server did not become healthy: {last_error}")


def start_server(model):
    env = os.environ.copy()
    env.update(COMMON_ENV)
    env["OLLAMA_CHAT_MODEL"] = model

    process = subprocess.Popen(
        [PYTHON, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", str(API_PORT)],
        cwd=PROJECT_ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    wait_for_health(BASE_URL)
    return process


def stop_server(process):
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=10)

## 비교 실행

In [ ]:
def ask(question):
    started = time.perf_counter()
    response = requests.post(
        f"{BASE_URL}/chat",
        json={"message": question},
        timeout=180,
    )
    elapsed = time.perf_counter() - started
    response.raise_for_status()
    data = response.json()
    return {
        "elapsed_sec": round(elapsed, 2),
        "answer": data.get("answer", ""),
        "sources": data.get("sources", []),
    }


all_results = []

for model in MODELS:
    print(f"\n=== Running model: {model} ===")
    server = start_server(model)
    try:
        for question in QUESTIONS:
            print(f"- {question}")
            try:
                result = ask(question)
                all_results.append({
                    "model": model,
                    "question": question,
                    **result,
                    "error": "",
                })
            except Exception as exc:
                all_results.append({
                    "model": model,
                    "question": question,
                    "elapsed_sec": None,
                    "answer": "",
                    "sources": [],
                    "error": str(exc),
                })
    finally:
        stop_server(server)

pprint(all_results[:2])

## 결과 저장

In [ ]:
timestamp = time.strftime("%Y%m%d-%H%M%S")
json_path = RESULT_DIR / f"model_comparison_{timestamp}.json"
csv_path = RESULT_DIR / f"model_comparison_{timestamp}.csv"

json_path.write_text(json.dumps(all_results, ensure_ascii=False, indent=2), encoding="utf-8")

with csv_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["model", "question", "elapsed_sec", "answer", "sources", "error"],
    )
    writer.writeheader()
    for row in all_results:
        writer.writerow({**row, "sources": json.dumps(row["sources"], ensure_ascii=False)})

print(json_path)
print(csv_path)

## 수동 평가표

각 항목은 1~5점으로 채운다.

In [ ]:
score_rows = []

for item in all_results:
    score_rows.append({
        "model": item["model"],
        "question": item["question"],
        "elapsed_sec": item["elapsed_sec"],
        "korean_naturalness": None,
        "grounding": None,
        "unknown_handling": None,
        "tourism_fit": None,
        "citation_quality": None,
        "speed_score": None,
        "notes": "",
    })

score_rows[:3]